# 数据准备
我们使用Milvus 文档 2.4.x中的常见问题页面作为 RAG 中的私有知识，这对于简单的 RAG 管道来说是一个很好的数据源。下载 zip 文件并将文档解压缩到milvus_docs 文件夹中。

下载链接：https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip

从`milvus_docs/en/faq` 文件夹中加载所有标记文件。对于每个文件，我们只需简单地使用 `#`来分隔文件中的内容，这样就能大致分隔出 markdown 文件中每个主要部分的内容。

In [1]:
from glob import glob

text_lines = []

for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

# 准备LLM和Embedding方法

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()

client = OpenAI(
    api_key=os.environ["SILICON_FLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1",
)

In [20]:
def emb_text(text):
    # 调用embedding模型
    response = client.embeddings.create(
        model="BAAI/bge-m3",  # 推荐使用的embedding模型
        input=text,
        encoding_format="float"  # 返回格式，可选"float"或"base64"
    )
    return response.data[0].embedding

In [26]:
len(emb_text("这是一个示例文本，用于生成embedding向量"))

1024

# Milvus准备

In [28]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(host='localhost', port=19530)

collection_name = 'rag_demo_collection'

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=1024, # 根据模型调整
    metric_type='IP', # Inner product distance
    consistency_level='Strong'
)

# 插入数据

In [29]:
from tqdm import tqdm

data = []

for i, line in enumerate(tqdm(text_lines, desc='Creating embeddings')):
    data.append({'id': i, 'vector':emb_text(line), 'text': line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|███████████████████████████████████████████████████| 72/72 [00:06<00:00, 11.37it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

# 构建RAG

In [44]:
# 指定问题，并在Collection中搜索该问题并检索语义前3个匹配项
import json

result = milvus_client.search(
    collection_name=collection_name,
    data = [emb_text(question)],
    limit = 3,
    search_params={'metric_type':'IP', 'params':{}},
    output_fields=['text']
)

# 返回原始文本和距离
retrieved_lines_with_distances = [
    (res['entity']['text'], res['distance']) for res in result[0]
]

# 
context = '\n'.join([line_with_distance[0] for line_with_distance in retrieved_lines_with_distances])

In [45]:
retrieved_lines_with_distances

[(' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###',
  0.7449795007705688),
 ("How does Milvus flush data?\n\nMilvus returns success when inserted data are loaded to the message queue. However, the data a

In [46]:
# 使用LLM获取模型结果
SYSTEM_PROMPT = """
You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""

USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.

<context>
{context}
</context>

<question>
{question}
</question>
"""


def ask_with_context(context: str, question: str):
    # 格式化用户提示词
    user_prompt = USER_PROMPT.format(context=context, question=question)

    # 调用OpenAI接口
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2  # 保持回答稳定
    )

    return response.choices[0].message.content

In [48]:
question = 'How is data stored in milvus?'

ask_with_context(context, question)

'In Milvus, data is stored in two main ways:\n\n1. **Inserted Data**: This includes vector data, scalar data, and collection-specific schema. It is stored in persistent storage as incremental logs. Milvus supports multiple object storage backends such as MinIO, AWS S3, Google Cloud Storage, Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage.\n\n2. **Metadata**: These are generated within Milvus and are stored in etcd, a distributed key-value store.\n\nSo, the inserted data are stored in external object storage systems, while metadata are stored in etcd.'